In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

DATA_DIR  = './'

customers  = pd.read_csv(DATA_DIR + 'customers.csv')
geography  = pd.read_csv(DATA_DIR + 'geography.csv')
inventory  = pd.read_csv(DATA_DIR + 'inventory.csv')
order_items  = pd.read_csv(DATA_DIR + 'order_items.csv')
orders     = pd.read_csv(DATA_DIR + 'orders.csv')
payments  = pd.read_csv(DATA_DIR + 'payments.csv')
products   = pd.read_csv(DATA_DIR + 'products.csv')
returns    = pd.read_csv(DATA_DIR + 'returns.csv')
reviews    = pd.read_csv(DATA_DIR + 'reviews.csv')
shipments  = pd.read_csv(DATA_DIR + 'shipments.csv')
promotions = pd.read_csv(DATA_DIR + 'promotions.csv')
sales  = pd.read_csv(DATA_DIR + 'sales.csv')
sample_submission  = pd.read_csv(DATA_DIR + 'sample_submission.csv')
web_traffic = pd.read_csv(DATA_DIR + 'web_traffic.csv')

FileNotFoundError: [Errno 2] No such file or directory: './customers.csv'

In [2]:
# 6.Trong customers.csv, xét các khách hàng có age_group khác null, nhóm tuổi nào có số đơn hàng trung bình trên mỗi khách hàng cao nhất? (tổng số đơn / số khách hàng trong nhóm)
merge_df = pd.merge(customers.dropna(subset=['age_group']), orders, on='customer_id', how = 'inner')

result_6 = (
    merge_df
    .groupby('age_group')
    .agg(total_orders=('order_id', 'count'),
         unique_customers=('customer_id', 'nunique')
    )
    .reset_index()
)

result_6['avg_order_per_customer'] = result_6['total_orders'] / result_6['unique_customers']
result_6 = result_6[['age_group', 'avg_order_per_customer']].sort_values('avg_order_per_customer')
print(result_6)
# Đáp án A

NameError: name 'customers' is not defined

In [3]:

#7. Vùng (region) nào trong geography.csv tạo ra tổng doanh thu cao nhất trong sales_train.csv?
merge_df = (order_items.merge(orders, on='order_id', how = 'inner').merge(geography, on='zip', how='inner'))

merge_df['revenue'] = (merge_df['quantity'] * merge_df['unit_price']) - merge_df['discount_amount']

result_7 = (
     merge_df
    .groupby('region')['revenue']
    .sum()
    .round(2)
    .reset_index(name='total_revenue')
    .sort_values('total_revenue', ascending=False)
)
print(result_7)
# Đáp án C

    region  total_revenue
1     East   7.291151e+09
0  Central   4.719491e+09
2     West   3.670227e+09


In [4]:

# 8. Trong các đơn hàng có order_status = ’cancelled’ trong orders.csv, phương thức thanh toán nào được sử dụng nhiều nhất?
result_8 = (
    orders[orders['order_status'] == 'cancelled']
    .groupby('payment_method')
    .size()
    .reset_index(name='buy')
    .sort_values('buy', ascending=False)
)
print(result_8)
# Đáp án A

  payment_method    buy
3    credit_card  28452
2            cod  15468
4         paypal   7817
0      apple_pay   5190
1  bank_transfer   2535


In [5]:
#9. Trong bốn kích thước sản phẩm (S, M, L, XL), kích thước nào có tỷ lệ trả hàng cao nhất, được định nghĩa là số bản ghi trong returns chia cho số dòng trong order_items (join với products theo product_id)?
merge = (order_items.merge(products, on='product_id',how='inner')
         .merge(returns, on=['order_id','product_id'],how = 'left', indicator=True))
merge['is_returned'] = (merge['_merge'] == 'both').astype(int)

result_9 = (
    merge
    .groupby('size')
    .agg(total_returns=('is_returned', 'sum'),total_items=('order_id', 'count'))
    .reset_index()
)

result_9['return_rate'] = result_9['total_returns'] / result_9['total_items']
result_9 = result_9[['size', 'return_rate']].sort_values('return_rate', ascending=False)
print(result_9)
#Đáp án A



  size  return_rate
2    S     0.056515
0    L     0.056250
1    M     0.055682
3   XL     0.055200


In [6]:
#10. Trong payments.csv, kế hoạch trả góp nào có giá trị thanh toán trung bình trên mỗi đơn hàng cao nhất?
result_10 = (
    payments
    .groupby('installments')['payment_value']
    .mean()
    .reset_index(name = 'avg_payment')
    .sort_values('avg_payment', ascending=False)
)
print(result_10)
# Đáp án C

   installments   avg_payment
3             6  24446.654403
2             3  24399.635486
4            12  24245.772694
0             1  24113.274166
1             2    708.473729
